# Downloading PlanetScope data (Planet API)

*Section 12 — Downloading data, notebook 7 of 8.* Previous: `06_copernicus_odata_sen2`. Next: `08_icepyx_download_icesat2`.

**You will learn:**

- how to authenticate with the Planet API using `rsgispy.dataaccess.planet_data_api`
- how to search for PSScene items by region, date and cloud cover
- how to create an order and download it when it is ready
- how to call the synchronous `run_*` functions from a Jupyter notebook

**Before you start:** a Planet account with data access (https://www.planet.com) and the `planet` Python package (`pip install planet`). Store your Planet username and password in an encoded credentials file (`planet_userinfo.txt`, see *Username and password* below), or use a Planet API key from the `PL_API_KEY` (or `RSGIS_PLANET_API_KEY`) environment variable.

**Data:** downloaded from Planet (Data and Orders APIs) into `outputs/`.

**Run time:** depends on the download; the search cells take about 1 minute (creating the order can then take minutes to hours).

> **Running this notebook:** every cell that logs in, queries the remote
> service or downloads data is tagged `skip-execution`, so the tutorial test
> runner does not run them. Run them yourself, in order, once you have an
> account and have created the credentials file.

This tutorial uses the Planet API to find and download PlanetScope data.

To run this tutorial you need access to the Planet API (https://www.planet.com).

## Differences between notebooks and scripts

The Planet Python API is asynchronous (it uses
[asyncio](https://docs.python.org/3/library/asyncio.html)). The rsgispy
`run_*` functions (`run_search_planet_items`, `run_create_planet_order`,
`run_download_planet_order`, ...) wrap the asynchronous calls with
`asyncio.run`, so in a normal Python script you simply call them:

```python
items = planet_data_api.run_search_planet_items(planet_auth, ...)
```

A Jupyter notebook already runs its own asyncio event loop, and two loops
cannot run in the same thread, so calling a `run_*` function directly in a
notebook raises an error. The small `run_in_thread` helper defined below runs
the function in a worker thread (which has no event loop) and returns its
result, so the same public functions work in both places.

*RSGISLib users:* the RSGISLib version of this notebook awaited the private
`_search_planet_items`, `_create_planet_order` and `_download_planet_order`
coroutines; use the public `run_*` functions instead.

## Import modules

In [1]:
import concurrent.futures
import datetime
import os

import rsgispy.tools.credentials
from rsgispy.dataaccess import planet_data_api

In [2]:
def run_in_thread(func, *args, **kwargs):
    """Run ``func`` in a worker thread and return its result.

    Needed in Jupyter for the rsgispy Planet ``run_*`` functions, which use
    ``asyncio.run``; in a plain Python script call them directly.
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(func, *args, **kwargs).result()

## Create output directory

In [3]:
out_dir_path = os.path.join("outputs", "planet_data")
os.makedirs(out_dir_path, exist_ok=True)

# The encoded username / password file (see below).
cred_file = "planet_userinfo.txt"

## Username and password

Never write your username and password into a notebook. rsgispy provides a
simple encoding of the username and password so that they are not stored as
plain text. Note that the encoding (base64) is **not** secure: keep the file
private and never commit it to version control.

Create the encoded file with the `rsgisuserpassfile` command-line tool, which
is installed with rsgispy and prompts for the password:

`rsgisuserpassfile -o planet_userinfo.txt -u <your Planet username>`

or, from Python, with
`rsgispy.tools.credentials.create_username_password_file(username, password, out_file)`.
Once you have created `planet_userinfo.txt` (in this folder, or change the path
below), read it into your notebook/script with `get_username_password`.
Alternatively, define the `RSGIS_PLANET_USER` and `RSGIS_PLANET_PASS` environment variables
and pass `None` for the username and password.

If you use a Planet API key instead, set the `PL_API_KEY` environment variable, skip the next cell and call `planet_data_api.planet_auth()` with no arguments.

*RSGISLib users:* the tool was `rsgisuserpassfile.py <file>`; the output file
is now given with `-o`.

In [ ]:
username, password = rsgispy.tools.credentials.get_username_password(
    input_file=cred_file
)

## 1. Get authentication object

When you use the API, you need to be authenticated. The function below will
create an authentication object using your login credentials.

In [ ]:
planet_auth = planet_data_api.planet_auth(username=username, password=password)

## 2. Search for scenes

The first step is to query to find the scenes you are interested in. In this
case, we are looking for scenes in May and June 2024 within a bounding box
(xMin, xMax, yMin, yMax) and with cloud cover of less than 10 percent.

The results have been restricted to just provide 2 scenes to speed up building
the order and the download. Given the small land area of the scenes you would
probably increase this.

In [ ]:
items = run_in_thread(
    planet_data_api.run_search_planet_items,
    planet_auth,
    item_type=planet_data_api.RSGIS_PLANET_ITEM_PSScene,
    bbox=(-4.1, -4.0, 52.4, 52.5),
    start_date=datetime.datetime(year=2024, month=5, day=1),
    end_date=datetime.datetime(year=2024, month=6, day=30),
    cloud_cover=10,
    sun_elevation_min=None,
    sun_elevation_max=None,
    view_angle_min=None,
    view_angle_max=None,
    max_n_rslts=2,
)

In [ ]:
items

## 3. Create an order

The next step is to create an order so the Planet servers will prepare the
data to be downloaded. You need to provide the list of items you found through
the search, specify the type of the items (which will match what you
specified in the search), and then the bundle you want to download (e.g.,
8-band surface reflectance).

By default, once the order has been created and is ready to download, Planet
will send you an email notification, although this can be turned off.

In [ ]:
order_info = run_in_thread(
    planet_data_api.run_create_planet_order,
    planet_auth,
    order_name="example_order",
    items=items,
    item_type=planet_data_api.RSGIS_PLANET_ITEM_PSScene,
    bundle_type=planet_data_api.RSGIS_PLANET_BUNDLE_analytic_8b_sr_udm2,
    email_notification=True,
)

Note, you will want to make a note of the order id — the `id` field in the
returned `order_info` dictionary. `planet_data_api.run_get_planet_orders(planet_auth)`
lists your orders if you need to find it again later.

In [ ]:
order_info

## 4. Download the order

Once the order has been submitted, you should be able to see it on the Planet
orders page, as shown below:

![Planet Order](figures/planet_orders_page.png)

Once the status has changed to `success` the data is available for download
and you can run the following function — note, if you run the function before
the data is ready you will get an error. It can take a little while for the
data to be available for download, so you might need some patience.

In [ ]:
dwnld_paths = run_in_thread(
    planet_data_api.run_download_planet_order,
    planet_auth,
    order_id=order_info["id"],
    out_file_path=out_dir_path,
    overwrite=False,
)

The download function will return a list of the file paths which have been downloaded:

In [ ]:
dwnld_paths